In [1]:


import os
import json
import urllib.request
import urllib.error

print('✅ All imports ready!')

✅ All imports ready!


In [ ]:


GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "api key here")

MODEL = "llama-3.3-70b-versatile"

chat_history = []

SYSTEM_PROMPT = """You are a helpful, friendly AI assistant.
You can answer questions on any topic — physics, math, coding, history, general knowledge, anything.
Keep answers clear and not too long unless the user asks for detail.
If you don't know something, say so honestly."""


In [3]:

def ask_groq(user_message):
    """Sends a message to Groq API and returns the AI's reply as a string."""
    
    chat_history.append({"role": "user", "content": user_message})
    
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}] + chat_history,
        "temperature": 0.7,   # 0 = precise/robotic, 1 = creative/random
        "max_tokens": 1024,   # max length of each reply
    }
    
    data = json.dumps(payload).encode("utf-8")
    
    req = urllib.request.Request(
        "https://api.groq.com/openai/v1/chat/completions",
        data=data,
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST"
    )
    
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            result = json.loads(response.read().decode("utf-8"))
            bot_reply = result["choices"][0]["message"]["content"]
            # Save reply to history so the bot remembers what it said
            chat_history.append({"role": "assistant", "content": bot_reply})
            return bot_reply
            
    except urllib.error.HTTPError as e:
        chat_history.pop()  # Remove the failed message from history
        if e.code == 401:
            return "❌ Invalid API key. Get a free one at https://console.groq.com/keys"
        elif e.code == 429:
            return "❌ Rate limit hit. Wait a moment and try again."
        else:
            return f"❌ API error {e.code}: {e.read().decode('utf-8')}"
    except urllib.error.URLError as e:
        chat_history.pop()  # Remove the failed message from history
        return f"❌ Network error — check your internet connection. ({e.reason})"
    except TimeoutError:
        chat_history.pop()
        return "❌ Request timed out. Try again."
    except Exception as e:
        chat_history.pop()
        return f"❌ Something went wrong: {str(e)}"


print('✅ ask_groq() is ready!')

✅ ask_groq() is ready!


In [5]:
# Cell 5: Follow-up question — tests context/memory

followup = "Give me a real life example of that."

print(f"You: {followup}")
print()
reply = ask_groq(followup)
print(f"Bot: {reply}")

You: Give me a real life example of that.

Bot: ❌ API error 403: error code: 1010


In [6]:
# Cell 6: Reset conversation memory

chat_history.clear()
print("🗑️ Chat history cleared — bot starts fresh now.")

🗑️ Chat history cleared — bot starts fresh now.


In [7]:


chat_history.clear()  

print("🤖 PyBot AI is ready! Ask me anything.")
print("   Type 'quit' to stop, 'clear' to reset memory\n")

while True:
    try:
        user_input = input("You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("Bot: Goodbye! 👋")
            break
        
        if user_input.lower() == "clear":
            chat_history.clear()
            print("🗑️ Memory cleared!\n")
            continue
        
        reply = ask_groq(user_input)
        print(f"\nBot: {reply}\n")
        
    except KeyboardInterrupt:
        print("\nBot: Goodbye! 👋")
        break

🤖 PyBot AI is ready! Ask me anything.
   Type 'quit' to stop, 'clear' to reset memory


Bot: ❌ API error 403: error code: 1010


Bot: ❌ API error 403: error code: 1010

Bot: Goodbye! 👋


In [8]:

MODEL = "llama-3.1-8b-instant"   
chat_history.clear()
print(f"✅ Switched to: {MODEL}")

q = "Explain quantum entanglement in simple terms."
print(f"\nYou: {q}\n")
print(f"Bot: {ask_groq(q)}")

✅ Switched to: llama-3.1-8b-instant

You: Explain quantum entanglement in simple terms.

Bot: ❌ API error 403: error code: 1010
